# Document summary agent

Regulators publish long documents, and compliance teams have to turn each one into something a decision-maker
can act on: what the firm must do, by when, and what it means for *this* firm. In this workshop you'll build a
**deep agent** that does this for a policy statement, using two agents:

1. A **main agent** (the orchestrator) loads the document, which arrives already split into sections, and looks up
   the firm's profile.
2. It hands each section to a **sub-agent**, `section-summariser`, which summarises that one section and extracts
   its obligations as a structured `SectionSummary`. The sub-agents run in parallel.
3. The main agent collects the summaries and writes a brief for the firm.

The regulator, the firm and the document are fictional, and stored in `data/`. The only network call is to the
model.

| Path | What's in it |
|------|--------------|
| `agent.py` | The main agent, exported as `agent`. |
| `instructions.md` | The main agent's system prompt. |
| `sub_agent.py` | The `section-summariser` sub-agent and the `SectionSummary` it returns. |
| `sub_agent_instructions.md` | The sub-agent's system prompt. |
| `tools/` | The lookups the agent uses: documents, firm profiles and providers. |
| `data/` | The firm's database: one JSONL file per table, plus the documents themselves. |

**How to use this notebook:** run each cell in order with Shift+Enter.

## Step 0: Setup

Copy `.env.example` to `.env` in the repo root (shared by all three topics) and fill in your gateway key. To use
another model, set `MODEL_NAME` as `provider:model` (for example `openai:gpt-4.1-mini`). The model is set up once,
in `utils/models.py`, so you don't need to edit any code to switch.

In [ ]:
# Notebook only: let Python find `agent.py`, `sub_agent.py` and the `utils/` and `tools/` folders.
# This notebook lives in modules/, so the workshop root is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from langchain.chat_models import init_chat_model

from utils.models import model

print(init_chat_model(model).invoke("Say hello to the compliance hackathon in one sentence.").text)

## Step 1: The tools

The data lives in `data/` as **JSONL** files: one file per table, one JSON object per line. `utils/db.py` reads
them. Two tools in `tools/` use them, and both return a JSON object, so the model gets clean, structured data:

- `get_document` returns the document's `document_id`, `title` and `sections`. Each section has its `number`,
  `title` and full `text`. We split at the `## ` headings, rather than every N characters, so each piece keeps its
  section number. We need those numbers later, to check every claim against its source.
- `get_firm_profile` returns the firm and its `third_party_providers`, each with its `contract_end` and whether it
  is `material`. The brief is for Harbour Mutual Bank, `FIRM-001`.

The shapes are defined as Pydantic models (`Document` and `Firm`) in those files.

In [ ]:
from tools.documents import get_document
from tools.firms import get_firm_profile

document = get_document("PS26-4")
print(document["title"], "\n")
for section in document["sections"]:
    print(f"{section['number']:>2}. {section['title']:<55} {len(section['text'].split()):>4} words")

In [ ]:
import json

print(json.dumps(get_firm_profile("FIRM-001"), indent=1))

## Step 2: The sub-agent

The sub-agent only ever sees one section, so it can focus on it. It is defined in `sub_agent.py` with three things:

- **Instructions** (`sub_agent_instructions.md`): what counts as an obligation, and the rules for accuracy.
- **A response format**: the `SectionSummary` Pydantic model. The model must fill it in, so every section comes
  back in the same shape.
- **A name and description**, which the main agent uses to decide when to call it.

The most important part of `SectionSummary` is the list of **obligations**: things the firm *must* do. For each
one, the model gives the paragraph number, the deadline exactly as written, and a quote copied word for word.
Those fields make the summary *reviewable*: a reviewer, or our code, can check each claim against the source.

In [ ]:
from sub_agent import INSTRUCTIONS as SUB_AGENT_INSTRUCTIONS, SectionSummary

print(SUB_AGENT_INSTRUCTIONS)

In [ ]:
print(json.dumps(SectionSummary.model_json_schema()["properties"], indent=1)[:900], "...")

Let's try the sub-agent on its own, on the incident reporting section. Here we wrap it in a deep agent of its own,
using the same instructions and response format, so we can see what it returns before the main agent uses it.

In [ ]:
from deepagents import create_deep_agent

summariser = create_deep_agent(
    model=init_chat_model(model),
    system_prompt=SUB_AGENT_INSTRUCTIONS,
    response_format=SectionSummary,
)

incident_reporting = document["sections"][6]
task = f"Section {incident_reporting['number']}: {incident_reporting['title']}\n\n{incident_reporting['text']}"
summary = summariser.invoke({"messages": [{"role": "user", "content": task}]})["structured_response"]

print(summary.summary, "\n")
for obligation in summary.obligations:
    print(f"- [{obligation.paragraph}] {obligation.requirement} (deadline: {obligation.deadline})")

## Step 3: Check the obligations with plain code

Models sometimes paraphrase when they should quote, or get a date slightly wrong. In a compliance brief, that's
not acceptable. So ordinary Python can check each obligation:

- Does its quote appear **word for word** in the section?
- Does its deadline appear in the section?

These checks are deterministic, so they're cheap, fast and easy to audit.

In [ ]:
import re


def normalise(text: str) -> str:
    """Lowercase, straighten quotes, collapse whitespace and trim punctuation, so formatting can't hide a match."""
    text = text.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", text).strip(" .\"'").lower()


def check_summary(summary: SectionSummary, section: dict) -> list[str]:
    """Return a list of problems with a section summary's obligations. An empty list means every one checks out."""
    section_text = normalise(section["text"])
    issues = []
    for i, obligation in enumerate(summary.obligations, start=1):
        if normalise(obligation.evidence) not in section_text:
            issues.append(f"Obligation {i}: the quote is not word for word from section {section['number']}.")
        elif obligation.deadline and normalise(obligation.deadline) not in section_text:
            issues.append(f"Obligation {i}: the deadline '{obligation.deadline}' is not in section {section['number']}.")
    return issues


print(check_summary(summary, incident_reporting) or "Every obligation checks out.")

## Step 4: The main agent

The main agent is defined in `agent.py`. It has:

- **Instructions** (`instructions.md`): load the document, give each section to its own `section-summariser`
  sub-agent in parallel, collect the summaries in section order, and combine them into a brief for the firm.
- **Tools**: `get_document` and `get_firm_profile`.
- **The sub-agent**: `section_summariser_subagent`, passed as `subagents`.

A deep agent is a complete harness, including a virtual filesystem and a `task` tool, which is how it calls a sub-agent.
That's what makes the fan-out possible without us writing any graph code.

In [ ]:
from agent import INSTRUCTIONS, agent

print(INSTRUCTIONS)

## Step 5: Run it

Give the main agent a document and a firm. Then we read back what happened: each `task` call is one run of the
`section-summariser` sub-agent. Notice that the model makes all ten calls in the same step. That's the parallel
fan-out.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Summarise document PS26-4 for FIRM-001"}]})

for message in result["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("called:", call["name"], call["args"].get("document_id") or call["args"].get("subagent_type") or "")

Each sub-agent returned a `SectionSummary` as JSON. Here is how many obligations it found in each section, and
whether they pass our check:

In [ ]:
summaries = [
    SectionSummary(**json.loads(message.content))
    for message in result["messages"]
    if message.type == "tool" and message.name == "task"
]
sections = {section["number"]: section for section in document["sections"]}

for s in sorted(summaries, key=lambda s: int(s.number)):
    problems = check_summary(s, sections[s.number])
    print(f"{s.number:>2}. {s.title:<55} {len(s.obligations):>2} obligations, {len(problems)} problems")

Here's the brief the main agent wrote:

In [ ]:
final = result["messages"][-1].text
print(final)

## Done! Where to look next

- **LangSmith:** if you set `LANGSMITH_TRACING=true` in `.env`, every run above is traced. Open it to see the ten
  sub-agent runs side by side, and every prompt the model received.
- **LangGraph Studio:** run `uv run langgraph dev` in the `03_document_summary_agent` folder and chat with the
  agent.
- **Managed Deep Agents:** run `uv run mda dev` to try the `agent` definition as a deep agent, and
  `uv run mda deploy` to deploy it. See the README.

## Stretch goals

1. **Presentation agent.** Separate the main agent so it returns raw data and passes it to a presentation agent for final report generation. Create datasets and experiements to allow for testing of data restrieval separate to presentation.
2. **Verify response within agent.** Give the agents a tool or middleware (e.g RubricMiddleware) that verifies the outputs and re-runs any
   sub-agent whose summary has problems.
3. **Human review.** Add human-in-the-loop, so a reviewer must approve the brief before it's final.
4. **Performanfe tuning.** Do all agents and sub-agents need the same model? Optimize for cost, performance and accuracy.
